# Spain Smart Meter Network Case Study

This notebook reproduces the key findings from the [Spain multi-meter experiment](../experiments/spain-multi-meter/) using the bundled results CSV — no 633M-row dataset required.

**What you'll learn:**
1. How HVG, NVG, and transition networks characterize smart-meter consumption
2. Why HVG average degree ≈ 4.0 holds at scale
3. How NVG variability reveals meter heterogeneity
4. How to use `NetworkFeatureExtractor` in a sklearn pipeline

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/kylejones200/ts2net/main?filepath=examples%2Fspain_meter_case_study.ipynb)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from ts2net.sklearn import NetworkFeatureExtractor

plt.style.use("seaborn-v0_8-whitegrid")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
CSV_PATH = REPO_ROOT / "experiments" / "spain-multi-meter" / "spain_meter_network_results.csv"
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} meters from {CSV_PATH.name}")

## 1. Published experiment results

The original experiment processed 50 meters from Spain's 633M-row smart-meter dataset in ~42 seconds. Network metrics were computed per meter after resampling to hourly consumption.

In [ ]:
summary = df[[
    "n_points", "hvg_avg_degree", "nvg_avg_degree", "tn_avg_degree", "std_consumption"
]].describe().round(2)
summary

**Key finding:** HVG average degree is consistently ~4.0 across diverse consumption patterns, validating the theoretical result at scale. NVG average degree varies widely (4.95–22.17), suggesting different consumption complexities.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df["hvg_avg_degree"], bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(4.0, color="red", linestyle="--", label="theory = 4.0")
axes[0].set_title("HVG average degree")
axes[0].set_xlabel("avg degree")
axes[0].legend()

axes[1].hist(df["nvg_avg_degree"], bins=20, color="darkorange", edgecolor="white")
axes[1].set_title("NVG average degree")
axes[1].set_xlabel("avg degree")

axes[2].scatter(df["n_points"], df["hvg_edges"], alpha=0.6, s=25)
axes[2].set_title("HVG edges vs series length")
axes[2].set_xlabel("hours")
axes[2].set_ylabel("edges")

plt.tight_layout()
plt.show()

## 2. Cluster meters by network signature

Group meters into low- vs high-complexity clusters using published network features.

In [ ]:
feature_cols = [
    "hvg_avg_degree", "nvg_avg_degree", "tn_avg_degree",
    "std_consumption", "mean_consumption",
]
X = df[feature_cols].values

km = KMeans(n_clusters=2, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(X)
print(f"Silhouette score: {silhouette_score(X, df['cluster']):.3f}")

cluster_nvg = df.groupby("cluster")["nvg_avg_degree"].mean().sort_values()
df.groupby("cluster")[["nvg_avg_degree", "std_consumption", "mean_consumption"]].mean().round(2)

## 3. sklearn pipeline with NetworkFeatureExtractor

The same workflow on raw time series: extract network features, then classify.

In [ ]:
rng = np.random.default_rng(42)
n_points = 400
series, labels = [], []

for label in range(2):
    for _ in range(25):
        t = np.arange(n_points)
        if label == 0:
            x = 0.3 + 0.1 * np.sin(2 * np.pi * t / 24) + 0.02 * rng.standard_normal(n_points)
        else:
            x = 1.5 + 0.4 * np.sin(2 * np.pi * t / 24) + 0.3 * rng.standard_normal(n_points)
            spikes = rng.choice(n_points, size=15, replace=False)
            x[spikes] += rng.uniform(2, 5, size=15)
        series.append(x)
        labels.append(label)

X_syn = np.vstack(series)
y_syn = np.array(labels)

pipe = Pipeline([
    ("net", NetworkFeatureExtractor(method="hvg", output="stats")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_syn, y_syn)
print(f"Training accuracy: {pipe.score(X_syn, y_syn):.3f}")
print("Features:", ", ".join(pipe.named_steps["net"].get_feature_names_out()))

## 4. Takeaways

- **HVG** gives stable, theory-validated features (~4.0 avg degree) ideal for large-scale screening
- **NVG** captures consumption heterogeneity — high-degree meters may warrant anomaly investigation
- **NetworkFeatureExtractor** bridges ts2net and sklearn for end-to-end ML pipelines

See `experiments/spain-multi-meter/README.md` for full experiment details and `examples/network_features_sklearn.py` for a runnable script version.